In [0]:
%sql
select * from retail_catalog.silver.sales_events_clean

In [0]:
%sql
select * from retail_catalog.bronze.sales_events

In [0]:
from pyspark.sql.functions import *

bronze_path = "/Volumes/retail_catalog/bronze/raw_sales/2/"
bronze_table = "retail_catalog.bronze.sales_events"

df = (spark.read.format("csv")
      .option("header", "true")
      .load(bronze_path)
      .withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", col("_metadata.file_path"))  # UC-safe way
)
df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_catalog.gold.pipeline_audit (
  pipeline_name STRING,
  last_run_ts TIMESTAMP
) USING DELTA;

INSERT INTO retail_catalog.gold.pipeline_audit 
VALUES ('scd2_sales_pipeline', '2025-09-01T00:00:00Z');

In [0]:
%sql
select * from retail_catalog.gold.pipeline_audit

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

bronze_df = spark.table("retail_catalog.bronze.sales_events")

# Deduplicate by keeping latest per order_id
w = Window.partitionBy("order_id", "customer_id").orderBy(col("ingest_ts").desc())
silver_df = (bronze_df
             .withColumn("rn", row_number().over(w))
             .filter(col("rn") == 1)
             .drop("rn"))

# Overwrite Silver snapshot each run
silver_df.write.format("delta").mode("overwrite").saveAsTable("retail_catalog.silver.sales_events_clean")

# audit table
audit_table = "retail_catalog.gold.pipeline_audit"
pipeline_name = "scd2_sales_pipeline"

last_run_ts = (spark.table(audit_table)
                .filter(f"pipeline_name = '{pipeline_name}'")
                .select("last_run_ts")
                .collect()[0][0])

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS retail_catalog.gold.pipeline_audit (
#   pipeline_name STRING,
#   last_run_ts TIMESTAMP
# ) USING DELTA;

In [0]:
from pyspark.sql.functions import *

silver_table = "retail_catalog.silver.sales_events_clean"

incremental_updates = (spark.table(silver_table)
                       .filter(col("ingest_ts") >= date_sub(current_date(), 3)))

incremental_updates.createOrReplaceTempView("updates")

In [0]:
%sql
ALTER TABLE retail_catalog.gold.dim_sales_scd2 ADD COLUMNS (customer_name STRING);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW updates AS
SELECT
  s.order_id,
  s.customer_id,
  s.customer_name,
  s.product_id,
  s.product_name,
  s.category,
  s.quantity,
  s.price,
  s.order_date,
  s.city,
  s.state,
  s.payment_type,
  s.ingest_ts,
  s.source_file,
  uuid() AS sales_sk,                          -- new surrogate key
  current_timestamp() AS effective_from,       -- start of validity
  NULL AS effective_to,                        -- end of validity
  true AS current_flag                         -- active row
FROM retail_catalog.silver.sales_events_clean s;


-- merge logic

MERGE INTO retail_catalog.gold.dim_sales_scd2 t
USING retail_catalog.silver.sales_events_clean s
ON t.order_id = s.order_id AND t.customer_id = s.customer_id AND t.current_flag = true
WHEN MATCHED AND (
    t.customer_name <> s.customer_name
 OR t.product_id <> s.product_id
 OR t.price <> s.price
 OR t.quantity <> s.quantity
)
  THEN UPDATE SET t.current_flag = false, t.effective_to = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
  sales_sk, order_id, customer_id, customer_name,
  product_id, product_name, category, quantity, price,
  order_date, city, state, payment_type,
  ingest_ts, source_file, effective_from, effective_to, current_flag
)
VALUES (
  uuid(), s.order_id, s.customer_id, s.customer_name,
  s.product_id, s.product_name, s.category, s.quantity, s.price,
  s.order_date, s.city, s.state, s.payment_type,
  s.ingest_ts, s.source_file, current_timestamp(), NULL, true
);

In [0]:
from pyspark.sql.functions import current_timestamp

spark.sql(f"""
MERGE INTO {audit_table} a
USING (SELECT '{pipeline_name}' as pipeline_name, current_timestamp() as new_ts) b
ON a.pipeline_name = b.pipeline_name
WHEN MATCHED THEN UPDATE SET a.last_run_ts = b.new_ts
WHEN NOT MATCHED THEN INSERT (pipeline_name, last_run_ts) VALUES (b.pipeline_name, b.new_ts)
""")

# test

In [0]:
%sql
SELECT * 
FROM retail_catalog.gold.dim_sales_scd2
WHERE order_id = '1001'
ORDER BY effective_from;

In [0]:
%sql
SELECT order_id, customer_name ,ingest_ts
FROM retail_catalog.bronze.sales_events
WHERE order_id = '1001'
ORDER BY ingest_ts DESC;

In [0]:
%sql
SELECT order_id, customer_name, ingest_ts
FROM retail_catalog.silver.sales_events_clean
WHERE order_id = '1001';

In [0]:
%sql
SELECT *
FROM retail_catalog.gold.dim_sales_scd2